# Stage 2 — ResNet-50 & MobileNetV2 baselines on Food-101

Fine-tunes two ImageNet-pretrained baselines on Food-101 for comparison with SHViT.

**Setup:** `Runtime → Change runtime type → T4 GPU` before running.

Per model, the training script writes:
- `checkpoints/<model>/training_log.csv` — one row per epoch (lr, losses, top-1/5)
- `checkpoints/<model>/best.pth` — checkpoint of highest val top-1

> **Heads-up:** 50 epochs of ResNet-50 on Food-101 at batch 64 takes a few hours on a free-tier T4. Drop `--epochs` if you just want to smoke-test the pipeline.

## 0. Verify GPU

In [ ]:
import torch
print('PyTorch     :', torch.__version__)
print('CUDA avail  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU         :', torch.cuda.get_device_name(0))

## 1. (Optional) Mount Drive

Persist the dataset and checkpoints across Colab sessions.

In [ ]:
USE_DRIVE = True

# Every file this notebook saves lives under a single root directory called
# CV_Research_Paper_Food101 (on Drive if USE_DRIVE, else on local Colab disk).
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/CV_Research_Paper_Food101'
else:
    BASE_DIR = '/content/CV_Research_Paper_Food101'

DATA_ROOT  = f'{BASE_DIR}/food101_data'
OUTPUT_DIR = f'{BASE_DIR}/Stage 2: baseline models'

import os
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Base       :', BASE_DIR)
print('Dataset    :', DATA_ROOT)
print('Checkpoints:', OUTPUT_DIR)

In [ ]:
# ---------------------------------------------------------------------------
# Robust Food-101 setup.
#
# The freezes/crashes come from heavy I/O on the Google Drive FUSE mount:
# downloading + extracting + copying the 101,000 small Food-101 image files
# through Drive is extremely slow and routinely stalls / kills the runtime.
#
# Fix: do ALL extraction and every training read on the LOCAL SSD, and persist
# only the single ~5 GB tarball on Drive (Drive handles one big file fine).
# torchvision verifies the tarball's MD5, so a corrupt/partial download is
# re-fetched cleanly instead of crashing later.
# ---------------------------------------------------------------------------
import os, shutil, time, torchvision

SPLIT_ID = '1QK0tGi096I0Ba6kggatX1ee6dJFIcEJl'

def setup_food101(local_data, drive_data=None):
    """Prepare Food-101 on local SSD; persist only the single ~5 GB tarball on
    Drive. Avoids extracting/copying 101k small files over the Drive FUSE mount
    (the cause of the freezes/crashes). Returns the local data root."""
    os.makedirs(local_data, exist_ok=True)
    local_tgz = f'{local_data}/food-101.tar.gz'
    drive_tgz = f'{drive_data}/food-101.tar.gz' if drive_data else None

    def have_images(root):
        d = f'{root}/food-101/images'
        return os.path.isdir(d) and len(os.listdir(d)) == 101

    # 1) Seed the local tarball from the Drive cache (one big-file copy is far
    #    more reliable than copying the extracted 101k-file tree).
    if (not have_images(local_data) and not os.path.exists(local_tgz)
            and drive_tgz and os.path.exists(drive_tgz)):
        print('Restoring cached food-101.tar.gz from Drive ...')
        t0 = time.time(); shutil.copy(drive_tgz, local_tgz)
        print(f'  done in {time.time()-t0:.0f}s')

    # 2) Download (if needed) + MD5-verify + extract, all LOCALLY.
    print('Preparing Food-101 locally (download/verify/extract) ...')
    t0 = time.time()
    torchvision.datasets.Food101(root=local_data, split='train', download=True)
    torchvision.datasets.Food101(root=local_data, split='test',  download=True)
    print(f'  ready in {time.time()-t0:.0f}s')
    assert have_images(local_data), 'Food-101 not fully extracted'

    # 3) Cache the verified tarball back to Drive for future sessions.
    if drive_tgz and not os.path.exists(drive_tgz) and os.path.exists(local_tgz):
        os.makedirs(drive_data, exist_ok=True)
        print('Caching tarball to Drive (one big file) ...')
        t0 = time.time(); shutil.copy(local_tgz, drive_tgz)
        print(f'  cached in {time.time()-t0:.0f}s')

    # 4) Zhou split file -> local food-101 dir (cache on Drive too).
    split_path  = f'{local_data}/food-101/split_zhou_Food101.json'
    drive_split = f'{drive_data}/food-101/split_zhou_Food101.json' if drive_data else None
    if not os.path.exists(split_path):
        if drive_split and os.path.exists(drive_split):
            shutil.copy(drive_split, split_path)
        else:
            os.system('pip install -q gdown')
            os.system(f'gdown "https://drive.google.com/uc?id={SPLIT_ID}" -O "{split_path}"')
            if drive_split:
                os.makedirs(os.path.dirname(drive_split), exist_ok=True)
                shutil.copy(split_path, drive_split)
    assert os.path.exists(split_path), 'split_zhou_Food101.json missing'
    print('Zhou split present:', split_path)
    return local_data

DATA_ROOT = setup_food101('/content/CV_Research_Paper_Food101/food101_data',
                          DATA_ROOT if USE_DRIVE else None)
print('DATA_ROOT now (local SSD):', DATA_ROOT)

## 2. Get the training script

Clone the project repo to pick up `train_baseline.py`.

In [ ]:
import os, shutil
REPO_DIR = '/content/Vision_Project_spring_26'
BRANCH   = 'Vision_Project_spring_26_Food101'
if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} \
        https://github.com/saif-farid-tech/Vision_Project_spring_26.git {REPO_DIR}

# Copy scripts + helper modules to /content for a clean working directory.
for fname in [
    'Stage 2: baseline models/train_baseline.py',
    'splits.py',
    'metrics.py',
    'augmentation.py',
]:
    shutil.copy(f'{REPO_DIR}/{fname}', f'/content/{os.path.basename(fname)}')
    print('Copied:', os.path.basename(fname))

# Copy the tip_datasets package (Tip-Adapter preprocessing + Zhou split loader).
if os.path.isdir('/content/tip_datasets'):
    shutil.rmtree('/content/tip_datasets')
shutil.copytree(f'{REPO_DIR}/tip_datasets', '/content/tip_datasets')
print('Copied: tip_datasets/')

## 3. Train ResNet-50

First run also downloads Food-101 (~5 GB, one-time).

In [ ]:
!python /content/train_baseline.py \
    --model resnet50 \
    --data-root "{DATA_ROOT}" \
    --output-dir "{OUTPUT_DIR}" \
    --epochs 50 \
    --batch-size 64 \
    --lr 1e-4 \
    --num-workers 2

## 4. Train MobileNetV2

In [ ]:
!python /content/train_baseline.py \
    --model mobilenet_v2 \
    --data-root "{DATA_ROOT}" \
    --output-dir "{OUTPUT_DIR}" \
    --epochs 50 \
    --batch-size 64 \
    --lr 1e-4 \
    --num-workers 2

## 5. Inspect logs

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for model_name in ['resnet50', 'mobilenet_v2']:
    csv_path = f'{OUTPUT_DIR}/{model_name}/training_log.csv'
    if not os.path.exists(csv_path):
        continue
    df = pd.read_csv(csv_path)
    axes[0].plot(df['epoch'], df['train_loss'], label=f'{model_name} train')
    axes[0].plot(df['epoch'], df['val_loss'],   label=f'{model_name} val', linestyle='--')
    axes[1].plot(df['epoch'], df['val_top1'] * 100, label=f'{model_name} top-1')
    axes[1].plot(df['epoch'], df['val_top5'] * 100, label=f'{model_name} top-5', linestyle='--')
    print(f'{model_name}: best val top-1 = {df["val_top1"].max()*100:.2f}%')

axes[0].set_title('Loss');     axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(True)
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()